# 01. 데이터 전처리

원본 탑승내역 데이터를 먼저 정제한 뒤, 정제된 데이터프레임을 기준으로 접수일시 파생 컬럼 CSV를 생성한다. 일별 이용현황 CSV 정리 코드는 별도 보관 섹션에서 실행한다.


In [8]:
import pandas as pd
from pathlib import Path

## 원본 탑승내역 데이터 불러오기

가장 기본이 되는 `서울시설공단_장애인콜택시 탑승내역_20251231.csv`를 먼저 불러온다. 이후 정제 작업은 이 데이터프레임에서 직접 반영한다.


In [9]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

input_path = PROJECT_ROOT / 'data' / 'raw' / '서울시설공단_장애인콜택시 탑승내역_20251231.csv'

df_raw = pd.read_csv(input_path)

print(f'원본 행 수: {len(df_raw):,}')
print(f'원본 컬럼 수: {df_raw.shape[1]:,}')
df_raw.head()

원본 행 수: 1,729,476
원본 컬럼 수: 15


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


## 논리적으로 맞지 않는 시간 데이터 점검

업무 흐름상 `접수일시 → 배차일시 → 승차일시 → 하차일시` 순서가 일반적이다. 먼저 `배차일시`가 `접수일시`보다 앞서는 데이터를 찾아 시간 순서가 맞지 않는 행을 확인한다.


In [10]:
# CHECK_DISPATCH_BEFORE_REQUEST
# 배차일시가 접수일시보다 앞서는 데이터 확인
request_dt = pd.to_datetime(df_raw['접수일시'], errors='coerce')
dispatch_dt = pd.to_datetime(df_raw['배차일시'], errors='coerce')

dispatch_before_request = df_raw[
    dispatch_dt.notna()
    & request_dt.notna()
    & (dispatch_dt < request_dt)
].copy()

dispatch_before_request.insert(0, '원본인덱스', dispatch_before_request.index)
dispatch_before_request.insert(1, 'CSV행번호', dispatch_before_request.index + 2)
dispatch_before_request['배차_접수_차이_분'] = (
    dispatch_dt.loc[dispatch_before_request.index] - request_dt.loc[dispatch_before_request.index]
).dt.total_seconds() / 60

print(f'배차일시가 접수일시보다 앞서는 행 수: {len(dispatch_before_request):,}')
dispatch_before_request[[
    '원본인덱스', 'CSV행번호', '접수일시', '예정일시', '배차일시',
    '승차일시', '하차일시', '취소일시', '배차_접수_차이_분'
]]


배차일시가 접수일시보다 앞서는 행 수: 1


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,배차_접수_차이_분
200,200,202,2025-01-01 08:34:51.000,2025-01-01 08:35:00.000,2025-01-01 08:30:00.000,2025-01-01 09:50:00.000,2025-01-01 10:07:00.000,NaN,-4.85


### index 200 제외

`배차일시 < 접수일시`로 확인된 pandas index `200` 행은 시간 순서가 논리적으로 맞지 않으므로, 처음 불러온 기본 데이터프레임 `df_raw`에서 제거한다. 원본 CSV 파일은 수정하지 않는다.


In [11]:
# EXCLUDE_INVALID_INDEX_200
# 원본 CSV는 수정하지 않고, 현재 노트북에서 불러온 기본 데이터프레임에서만 pandas index 200 제외
invalid_time_order_indices = [200]

before_rows = len(df_raw)
df_raw = df_raw.drop(index=invalid_time_order_indices, errors='ignore')
after_rows = len(df_raw)

print(f'제외 전 행 수: {before_rows:,}')
print(f'제외 후 행 수: {after_rows:,}')
print(f'제외된 행 수: {before_rows - after_rows:,}')

df_raw.head()


제외 전 행 수: 1,729,476
제외 후 행 수: 1,729,475
제외된 행 수: 1


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


### 승차일시 < 접수일시 데이터 확인

`승차일시`가 `접수일시`보다 앞서는 행은 접수 전에 탑승한 것으로 기록된 경우이므로, 시간 순서가 논리적으로 맞지 않는 데이터인지 확인한다.


In [12]:
# CHECK_RIDE_BEFORE_REQUEST
# 승차일시가 접수일시보다 앞서는 데이터 확인
request_dt = pd.to_datetime(df_raw['접수일시'], errors='coerce')
ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')

ride_before_request = df_raw[
    ride_dt.notna()
    & request_dt.notna()
    & (ride_dt < request_dt)
].copy()

ride_before_request.insert(0, '원본인덱스', ride_before_request.index)
ride_before_request.insert(1, 'CSV행번호', ride_before_request.index + 2)
ride_before_request['승차_접수_차이_분'] = (
    ride_dt.loc[ride_before_request.index] - request_dt.loc[ride_before_request.index]
).dt.total_seconds() / 60

print(f'승차일시가 접수일시보다 앞서는 행 수: {len(ride_before_request):,}')

ride_before_request[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '승차_접수_차이_분'
    ]
]


승차일시가 접수일시보다 앞서는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,승차_접수_차이_분


### 하차일시 < 접수일시 데이터 확인

`하차일시`가 `접수일시`보다 앞서는 행은 접수 전에 운행이 종료된 것으로 기록된 경우이므로, 시간 순서가 논리적으로 맞지 않는 데이터인지 확인한다.


In [13]:
# CHECK_ALIGHT_BEFORE_REQUEST
# 하차일시가 접수일시보다 앞서는 데이터 확인
request_dt = pd.to_datetime(df_raw['접수일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

alight_before_request = df_raw[
    alight_dt.notna()
    & request_dt.notna()
    & (alight_dt < request_dt)
].copy()

alight_before_request.insert(0, '원본인덱스', alight_before_request.index)
alight_before_request.insert(1, 'CSV행번호', alight_before_request.index + 2)
alight_before_request['하차_접수_차이_분'] = (
    alight_dt.loc[alight_before_request.index] - request_dt.loc[alight_before_request.index]
).dt.total_seconds() / 60

print(f'하차일시가 접수일시보다 앞서는 행 수: {len(alight_before_request):,}')

alight_before_request[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '하차_접수_차이_분'
    ]
]


하차일시가 접수일시보다 앞서는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,하차_접수_차이_분


### 취소일시 < 접수일시 데이터 확인

`취소일시`가 `접수일시`보다 앞서는 행을 확인한다. 이 경우는 예약 접수나 사전 취소처럼 업무적으로 설명 가능한 경우가 있을 수 있으므로, 바로 삭제하지 않고 별도 검토 대상으로 분리한다.


In [14]:
# CHECK_CANCEL_BEFORE_REQUEST
# 취소일시가 접수일시보다 앞서는 데이터 확인
request_dt = pd.to_datetime(df_raw['접수일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

cancel_before_request = df_raw[
    cancel_dt.notna()
    & request_dt.notna()
    & (cancel_dt < request_dt)
].copy()

cancel_before_request.insert(0, '원본인덱스', cancel_before_request.index)
cancel_before_request.insert(1, 'CSV행번호', cancel_before_request.index + 2)
cancel_before_request['취소_접수_차이_분'] = (
    cancel_dt.loc[cancel_before_request.index] - request_dt.loc[cancel_before_request.index]
).dt.total_seconds() / 60

print(f'취소일시가 접수일시보다 앞서는 행 수: {len(cancel_before_request):,}')

cancel_before_request[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '취소_접수_차이_분'
    ]
]


취소일시가 접수일시보다 앞서는 행 수: 6


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,취소_접수_차이_분
107,107,109,2025-01-01 04:45:00.737,2025-01-01 06:50:00.000,NaN,NaN,NaN,2024-01-01 09:00:00.000,-526785.012283
887622,887622,887624,2025-07-11 19:11:53.000,2025-07-11 19:12:00.000,NaN,NaN,NaN,2025-07-11 17:51:05.000,-80.800000
887818,887818,887820,2025-07-12 00:42:14.643,2025-07-12 00:42:14.643,NaN,NaN,NaN,2025-07-11 17:47:15.000,-414.994050
887819,887819,887821,2025-07-12 00:51:30.000,2025-07-12 00:52:00.000,NaN,NaN,NaN,2025-07-11 17:50:36.000,-420.900000
887826,887826,887828,2025-07-11 23:55:00.383,2025-07-12 02:00:00.000,NaN,NaN,NaN,2025-07-11 17:50:46.000,-364.239717
887838,887838,887840,2025-07-12 03:10:36.950,2025-07-12 03:10:36.950,NaN,NaN,NaN,2025-07-11 18:16:17.000,-534.332500


### 취소일시 < 접수일시 6개 행 제외

`취소일시`가 `접수일시`보다 앞서는 6개 행은 분석 기준상 논리적으로 맞지 않는 시간 데이터로 판단해, 원본 CSV는 수정하지 않고 현재 노트북의 `df_raw`에서만 제외한다.

In [15]:
# EXCLUDE_CANCEL_BEFORE_REQUEST
# 취소일시가 접수일시보다 앞서는 6개 행을 현재 df_raw에서 제외

cancel_before_request_indices = cancel_before_request.index

if len(cancel_before_request_indices) != 6:
    raise ValueError(f'제외 대상 행 수가 예상과 다릅니다: {len(cancel_before_request_indices):,}행')

before_rows = len(df_raw)
df_raw = df_raw.drop(index=cancel_before_request_indices, errors='ignore')
after_rows = len(df_raw)

request_dt = pd.to_datetime(df_raw['접수일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')
remaining_cancel_before_request = df_raw[
    cancel_dt.notna()
    & request_dt.notna()
    & (cancel_dt < request_dt)
]

print(f'제외 전 행 수: {before_rows:,}')
print(f'제외 후 행 수: {after_rows:,}')
print(f'제외된 행 수: {before_rows - after_rows:,}')
print(f'남은 취소일시 < 접수일시 행 수: {len(remaining_cancel_before_request):,}')

df_raw.head()

제외 전 행 수: 1,729,475
제외 후 행 수: 1,729,469
제외된 행 수: 6
남은 취소일시 < 접수일시 행 수: 0


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


### 승차일시 < 배차일시 데이터 확인

`승차일시`가 `배차일시`보다 앞서는 행은 배차 전에 탑승한 것으로 기록된 경우이므로, 시간 순서가 논리적으로 맞지 않는 데이터인지 확인한다.

In [16]:
# CHECK_RIDE_BEFORE_DISPATCH
# 승차일시가 배차일시보다 앞서는 데이터 확인

dispatch_dt = pd.to_datetime(df_raw['배차일시'], errors='coerce')
ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')

ride_before_dispatch = df_raw[
    ride_dt.notna()
    & dispatch_dt.notna()
    & (ride_dt < dispatch_dt)
].copy()

ride_before_dispatch.insert(0, '원본인덱스', ride_before_dispatch.index)
ride_before_dispatch.insert(1, 'CSV행번호', ride_before_dispatch.index + 2)
ride_before_dispatch['승차_배차_차이_분'] = (
    ride_dt.loc[ride_before_dispatch.index] - dispatch_dt.loc[ride_before_dispatch.index]
).dt.total_seconds() / 60

print(f'승차일시가 배차일시보다 앞서는 행 수: {len(ride_before_dispatch):,}')

ride_before_dispatch[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '승차_배차_차이_분'
    ]
]

승차일시가 배차일시보다 앞서는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,승차_배차_차이_분


### 하차일시 < 배차일시 데이터 확인

`하차일시`가 `배차일시`보다 앞서는 행은 배차 전에 운행이 종료된 것으로 기록된 경우이므로, 시간 순서가 논리적으로 맞지 않는 데이터인지 확인한다.

In [17]:
# CHECK_ALIGHT_BEFORE_DISPATCH
# 하차일시가 배차일시보다 앞서는 데이터 확인

dispatch_dt = pd.to_datetime(df_raw['배차일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

alight_before_dispatch = df_raw[
    alight_dt.notna()
    & dispatch_dt.notna()
    & (alight_dt < dispatch_dt)
].copy()

alight_before_dispatch.insert(0, '원본인덱스', alight_before_dispatch.index)
alight_before_dispatch.insert(1, 'CSV행번호', alight_before_dispatch.index + 2)
alight_before_dispatch['하차_배차_차이_분'] = (
    alight_dt.loc[alight_before_dispatch.index] - dispatch_dt.loc[alight_before_dispatch.index]
).dt.total_seconds() / 60

print(f'하차일시가 배차일시보다 앞서는 행 수: {len(alight_before_dispatch):,}')

alight_before_dispatch[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '하차_배차_차이_분'
    ]
]

하차일시가 배차일시보다 앞서는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,하차_배차_차이_분


### 취소일시 < 배차일시 데이터 확인

`취소일시`가 `배차일시`보다 앞서는 행은 배차 전에 취소된 것으로 기록된 경우이므로, 예약 취소나 기록 순서상 설명 가능한 경우인지 별도 확인한다.

In [18]:
# CHECK_CANCEL_BEFORE_DISPATCH
# 취소일시가 배차일시보다 앞서는 데이터 확인

dispatch_dt = pd.to_datetime(df_raw['배차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

cancel_before_dispatch = df_raw[
    cancel_dt.notna()
    & dispatch_dt.notna()
    & (cancel_dt < dispatch_dt)
].copy()

cancel_before_dispatch.insert(0, '원본인덱스', cancel_before_dispatch.index)
cancel_before_dispatch.insert(1, 'CSV행번호', cancel_before_dispatch.index + 2)
cancel_before_dispatch['취소_배차_차이_분'] = (
    cancel_dt.loc[cancel_before_dispatch.index] - dispatch_dt.loc[cancel_before_dispatch.index]
).dt.total_seconds() / 60

print(f'취소일시가 배차일시보다 앞서는 행 수: {len(cancel_before_dispatch):,}')

cancel_before_dispatch[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '취소_배차_차이_분'
    ]
]

취소일시가 배차일시보다 앞서는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,취소_배차_차이_분


### 하차일시 < 승차일시 데이터 확인

`하차일시`가 `승차일시`보다 앞서는 행은 승차 전에 운행이 종료된 것으로 기록된 경우이므로, 시간 순서가 논리적으로 맞지 않는 데이터인지 확인한다.

In [19]:
# CHECK_ALIGHT_BEFORE_RIDE
# 하차일시가 승차일시보다 앞서는 데이터 확인

ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

alight_before_ride = df_raw[
    alight_dt.notna()
    & ride_dt.notna()
    & (alight_dt < ride_dt)
].copy()

alight_before_ride.insert(0, '원본인덱스', alight_before_ride.index)
alight_before_ride.insert(1, 'CSV행번호', alight_before_ride.index + 2)
alight_before_ride['하차_승차_차이_분'] = (
    alight_dt.loc[alight_before_ride.index] - ride_dt.loc[alight_before_ride.index]
).dt.total_seconds() / 60

print(f'하차일시가 승차일시보다 앞서는 행 수: {len(alight_before_ride):,}')

alight_before_ride[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '하차_승차_차이_분'
    ]
]

하차일시가 승차일시보다 앞서는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,하차_승차_차이_분


### 취소일시 < 승차일시 데이터 확인

`취소일시`가 `승차일시`보다 앞서는 행은 승차 전에 취소된 것으로 기록된 경우이므로, 실제 탑승 여부와 취소 기록의 관계를 별도 확인한다.

In [20]:
# CHECK_CANCEL_BEFORE_RIDE

# 취소일시가 승차일시보다 앞서는 데이터 확인

ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

cancel_before_ride = df_raw[
    cancel_dt.notna()
    & ride_dt.notna()
    & (cancel_dt < ride_dt)
].copy()

cancel_before_ride.insert(0, '원본인덱스', cancel_before_ride.index)
cancel_before_ride.insert(1, 'CSV행번호', cancel_before_ride.index + 2)
cancel_before_ride['취소_승차_차이_분'] = (
    cancel_dt.loc[cancel_before_ride.index] - ride_dt.loc[cancel_before_ride.index]
).dt.total_seconds() / 60

print(f'취소일시가 승차일시보다 앞서는 행 수: {len(cancel_before_ride):,}')

cancel_before_ride[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '취소_승차_차이_분'
    ]
]

취소일시가 승차일시보다 앞서는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,취소_승차_차이_분


### 취소일시 < 하차일시 데이터 확인

`취소일시`가 `하차일시`보다 앞서는 행은 운행 종료 전에 취소된 것으로 기록된 경우이므로, 실제 운행 기록과 취소 기록의 관계를 별도 확인한다.

In [21]:
# CHECK_CANCEL_BEFORE_ALIGHT
# 취소일시가 하차일시보다 앞서는 데이터 확인

alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

cancel_before_alight = df_raw[
    cancel_dt.notna()
    & alight_dt.notna()
    & (cancel_dt < alight_dt)
].copy()

cancel_before_alight.insert(0, '원본인덱스', cancel_before_alight.index)
cancel_before_alight.insert(1, 'CSV행번호', cancel_before_alight.index + 2)
cancel_before_alight['취소_하차_차이_분'] = (
    cancel_dt.loc[cancel_before_alight.index] - alight_dt.loc[cancel_before_alight.index]
).dt.total_seconds() / 60

print(f'취소일시가 하차일시보다 앞서는 행 수: {len(cancel_before_alight):,}')

cancel_before_alight[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '취소_하차_차이_분'
    ]
]

취소일시가 하차일시보다 앞서는 행 수: 1


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,취소_하차_차이_분
168,168,170,2024-12-31 19:07:09.037,2025-01-01 08:00:00.000,2025-01-01 08:26:00.000,2025-01-01 08:49:00.000,2025-01-01 09:21:00.000,2025-01-01 09:16:34.000,-4.433333


 ### 취소일시 < 하차일시 행 제외

`취소일시`가 `하차일시`보다 앞서는 행은 운행 종료 전에 취소된 것으로 기록된 데이터이므로, 분석 기준상 논리적으로 맞지 않는 시간 데이터로 판단해 현재 노트북의 `df_raw`에서 제외한다. 원본 CSV 파일은 수정하지 않는다.

In [22]:
# EXCLUDE_CANCEL_BEFORE_ALIGHT

# 취소일시가 하차일시보다 앞서는 행을 현재 df_raw에서 제외

cancel_before_alight_indices = cancel_before_alight.index

before_rows = len(df_raw)
df_raw = df_raw.drop(index=cancel_before_alight_indices, errors='ignore')
after_rows = len(df_raw)

alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

remaining_cancel_before_alight = df_raw[
    cancel_dt.notna()
    & alight_dt.notna()
    & (cancel_dt < alight_dt)
]

print(f'제외 전 행 수: {before_rows:,}')
print(f'제외 후 행 수: {after_rows:,}')
print(f'제외된 행 수: {before_rows - after_rows:,}')
print(f'남은 취소일시 < 하차일시 행 수: {len(remaining_cancel_before_alight):,}')

df_raw.head()

제외 전 행 수: 1,729,469
제외 후 행 수: 1,729,468
제외된 행 수: 1
남은 취소일시 < 하차일시 행 수: 0


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


### 승차일시와 취소일시가 모두 있는 데이터 확인

`승차일시`와 `취소일시`가 모두 있는 행은 실제 탑승 기록과 취소 기록이 함께 존재하는 경우이므로, 취소가 탑승 전후 어느 시점에 발생했는지와 분석 제외 필요 여부를 확인한다.|

In [23]:
# CHECK_RIDE_AND_CANCEL_EXISTS

# 승차일시와 취소일시가 모두 있는 데이터 확인

ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

ride_and_cancel_exists = df_raw[
    ride_dt.notna()
    & cancel_dt.notna()
].copy()

ride_and_cancel_exists.insert(0, '원본인덱스', ride_and_cancel_exists.index)
ride_and_cancel_exists.insert(1, 'CSV행번호', ride_and_cancel_exists.index + 2)
ride_and_cancel_exists['취소_승차_차이_분'] = (
    cancel_dt.loc[ride_and_cancel_exists.index] - ride_dt.loc[ride_and_cancel_exists.index]
).dt.total_seconds() / 60

print(f'승차일시와 취소일시가 모두 있는 행 수: {len(ride_and_cancel_exists):,}')

ride_and_cancel_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '취소_승차_차이_분'
    ]
]

승차일시와 취소일시가 모두 있는 행 수: 33


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,취소_승차_차이_분
171,171,173,2024-12-31 07:00:31.000,2025-01-01 08:01:00.000,2025-01-01 08:26:00.000,2025-01-01 08:49:00.000,2025-01-01 09:21:00.000,2025-01-01 10:11:59.000,3000.0,12000,82.983333
28972,28972,28974,2025-01-08 09:33:51.670,2025-01-08 09:33:51.670,2025-01-08 09:41:08.043,2025-01-08 09:57:38.460,NaN,2025-01-08 09:59:02.000,NaN,0,1.392333
43539,43539,43541,2025-01-10 17:46:53.990,2025-01-10 17:46:53.990,2025-01-10 17:49:18.503,2025-01-10 17:52:21.783,NaN,2025-01-10 17:52:49.000,NaN,0,0.453617
62067,62067,62069,2025-01-15 12:31:50.440,2025-01-15 12:31:50.440,2025-01-15 12:39:20.267,2025-01-15 13:04:45.083,NaN,2025-01-15 13:19:16.000,NaN,0,14.515283
79982,79982,79984,2025-01-19 17:01:41.857,2025-01-19 17:01:41.857,2025-01-19 17:13:52.740,2025-01-19 17:49:06.250,NaN,2025-01-19 19:49:40.000,NaN,0,120.562500
165199,165199,165201,2025-02-10 10:25:00.807,2025-02-10 12:30:00.000,2025-02-10 12:34:29.730,2025-02-10 12:52:18.140,NaN,2025-02-10 12:57:48.000,NaN,0,5.497667
338598,338598,338600,2025-03-19 12:16:28.830,2025-03-19 12:16:28.830,2025-03-19 12:25:36.457,2025-03-19 12:37:34.023,NaN,2025-03-19 12:52:44.000,NaN,0,15.166283
387809,387809,387811,2025-03-29 12:07:33.403,2025-03-29 12:07:33.403,2025-03-29 12:26:38.170,2025-03-29 12:28:17.190,NaN,2025-03-29 12:28:24.000,NaN,0,0.113500
400215,400215,400217,2025-04-01 13:45:45.137,2025-04-01 13:45:45.137,2025-04-01 13:50:11.500,2025-04-01 13:58:04.657,NaN,2025-04-01 14:03:04.000,NaN,0,4.989050
414413,414413,414415,2025-04-03 07:00:08.930,2025-04-04 07:00:00.000,2025-04-04 07:04:47.707,2025-04-04 07:13:15.077,NaN,2025-04-04 07:24:02.000,NaN,0,10.782050


### 승차일시와 취소일시가 모두 있는 행 제외

`승차일시`와 `취소일시`가 모두 있는 행은 실제 탑승 기록과 취소 기록이 함께 존재하는 데이터이므로, 분석 기준상 상태가 모호한 행으로 판단해 현재 노트북의 `df_raw`에서 제외한다. 원본 CSV 파일은 수정하지 않는다.

In [24]:
# EXCLUDE_RIDE_AND_CANCEL_EXISTS

# 승차일시와 취소일시가 모두 있는 행을 현재 df_raw에서 제외

ride_and_cancel_exists_indices = ride_and_cancel_exists.index

before_rows = len(df_raw)
df_raw = df_raw.drop(index=ride_and_cancel_exists_indices, errors='ignore')
after_rows = len(df_raw)

ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

remaining_ride_and_cancel_exists = df_raw[
    ride_dt.notna()
    & cancel_dt.notna()
]

print(f'제외 전 행 수: {before_rows:,}')
print(f'제외 후 행 수: {after_rows:,}')
print(f'제외된 행 수: {before_rows - after_rows:,}')
print(f'남은 승차일시+취소일시 동시 존재 행 수: {len(remaining_ride_and_cancel_exists):,}')

df_raw.head()

제외 전 행 수: 1,729,468
제외 후 행 수: 1,729,435
제외된 행 수: 33
남은 승차일시+취소일시 동시 존재 행 수: 0


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


### 하차일시와 취소일시가 모두 있는 데이터 확인

`하차일시`와 `취소일시`가 모두 있는 행은 운행 종료 기록과 취소 기록이 함께 존재하는 경우이므로, 실제 운행 완료 여부와 취소 기록의 관계를 확인한다.

In [25]:
# CHECK_ALIGHT_AND_CANCEL_EXISTS
# 하차일시와 취소일시가 모두 있는 데이터 확인

alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

alight_and_cancel_exists = df_raw[
    alight_dt.notna()
    & cancel_dt.notna()
].copy()

alight_and_cancel_exists.insert(0, '원본인덱스', alight_and_cancel_exists.index)
alight_and_cancel_exists.insert(1, 'CSV행번호', alight_and_cancel_exists.index + 2)
alight_and_cancel_exists['취소_하차_차이_분'] = (
    cancel_dt.loc[alight_and_cancel_exists.index] - alight_dt.loc[alight_and_cancel_exists.index]
).dt.total_seconds() / 60

print(f'하차일시와 취소일시가 모두 있는 행 수: {len(alight_and_cancel_exists):,}')

alight_and_cancel_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '취소_하차_차이_분'
    ]
]

하차일시와 취소일시가 모두 있는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,취소_하차_차이_분


### 취소일시와 요금이 모두 있는 데이터 확인

`취소일시`와 `요금`이 모두 있는 행은 취소 기록이 있으면서 요금도 기록된 경우이므로, 실제 운행 여부와 결제 기록의 관계를 확인한다.

In [26]:
# CHECK_CANCEL_AND_FARE_EXISTS
# 취소일시와 요금이 모두 있는 데이터 확인

cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')

cancel_and_fare_exists = df_raw[
    cancel_dt.notna()
    & fare.notna()
].copy()

cancel_and_fare_exists.insert(0, '원본인덱스', cancel_and_fare_exists.index)
cancel_and_fare_exists.insert(1, 'CSV행번호', cancel_and_fare_exists.index + 2)

print(f'취소일시와 요금이 모두 있는 행 수: {len(cancel_and_fare_exists):,}')

cancel_and_fare_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

취소일시와 요금이 모두 있는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 취소일시가 있고 승차거리가 0보다 큰 데이터 확인

`취소일시`가 있으면서 `승차거리`가 0보다 큰 행은 취소 기록이 있으나 실제 이동 거리가 기록된 경우이므로, 실제 운행 여부와 취소 기록의 관계를 확인한다.

In [27]:
# CHECK_CANCEL_AND_POSITIVE_DISTANCE

# 취소일시가 있고 승차거리가 0보다 큰 데이터 확인

cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')
ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')

cancel_and_positive_distance = df_raw[
    cancel_dt.notna()
    & (ride_distance > 0)
].copy()

cancel_and_positive_distance.insert(0, '원본인덱스', cancel_and_positive_distance.index)
cancel_and_positive_distance.insert(1, 'CSV행번호', cancel_and_positive_distance.index + 2)

print(f'취소일시가 있고 승차거리가 0보다 큰 행 수: {len(cancel_and_positive_distance):,}')

cancel_and_positive_distance[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

취소일시가 있고 승차거리가 0보다 큰 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 하차일시가 없고 요금이 있는 데이터 확인

`하차일시`가 비어 있는데 `요금`이 있는 행은 운행 종료 시각 없이 요금만 기록된 경우이므로, 실제 운행 완료 여부와 요금 기록의 관계를 확인한다.

In [28]:
# CHECK_MISSING_ALIGHT_AND_FARE_EXISTS

# 하차일시가 없고 요금이 있는 데이터 확인

alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')

missing_alight_and_fare_exists = df_raw[
    alight_dt.isna()
    & fare.notna()
].copy()

missing_alight_and_fare_exists.insert(0, '원본인덱스', missing_alight_and_fare_exists.index)
missing_alight_and_fare_exists.insert(1, 'CSV행번호', missing_alight_and_fare_exists.index + 2)

print(f'하차일시가 없고 요금이 있는 행 수: {len(missing_alight_and_fare_exists):,}')

missing_alight_and_fare_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

하차일시가 없고 요금이 있는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 하차일시가 없고 승차거리가 0보다 큰 데이터 확인

`하차일시`가 비어 있는데 `승차거리`가 0보다 큰 행은 운행 종료 시각 없이 실제 이동 거리가 기록된 경우이므로, 운행 완료 여부와 거리 기록의 관계를 확인한다.

In [29]:
# CHECK_MISSING_ALIGHT_AND_POSITIVE_DISTANCE

# 하차일시가 없고 승차거리가 0보다 큰 데이터 확인

alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')

missing_alight_and_positive_distance = df_raw[
    alight_dt.isna()
    & (ride_distance > 0)
].copy()

missing_alight_and_positive_distance.insert(0, '원본인덱스', missing_alight_and_positive_distance.index)
missing_alight_and_positive_distance.insert(1, 'CSV행번호', missing_alight_and_positive_distance.index + 2)

print(f'하차일시가 없고 승차거리가 0보다 큰 행 수: {len(missing_alight_and_positive_distance):,}')

missing_alight_and_positive_distance[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

하차일시가 없고 승차거리가 0보다 큰 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 승차일시가 없고 하차일시가 있는 데이터 확인

`승차일시`가 비어 있는데 `하차일시`가 있는 행은 탑승 시작 시각 없이 운행 종료 시각만 기록된 경우이므로, 시간 기록 누락 여부와 실제 운행 완료 여부를 확인한다.

In [30]:
# CHECK_MISSING_RIDE_AND_ALIGHT_EXISTS

# 승차일시가 없고 하차일시가 있는 데이터 확인

ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

missing_ride_and_alight_exists = df_raw[
    ride_dt.isna()
    & alight_dt.notna()
].copy()

missing_ride_and_alight_exists.insert(0, '원본인덱스', missing_ride_and_alight_exists.index)
missing_ride_and_alight_exists.insert(1, 'CSV행번호', missing_ride_and_alight_exists.index + 2)

print(f'승차일시가 없고 하차일시가 있는 행 수: {len(missing_ride_and_alight_exists):,}')

missing_ride_and_alight_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

승차일시가 없고 하차일시가 있는 행 수: 691


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리
362,362,364,2025-01-01 10:00:59.000,2025-01-01 10:02:00.000,2025-01-01 10:24:02.627,NaN,2025-01-01 13:58:08.010,NaN,0.0,0
3647,3647,3649,2025-01-02 11:55:23.923,2025-01-02 11:55:00.000,2025-01-02 12:08:51.623,NaN,2025-01-02 13:41:51.887,NaN,0.0,0
4111,4111,4113,2025-01-02 12:49:37.000,2025-01-02 12:50:00.000,2025-01-02 13:36:46.540,NaN,2025-01-02 13:42:16.403,NaN,0.0,0
9200,9200,9202,2025-01-03 12:16:52.780,2025-01-03 12:17:00.000,2025-01-03 12:18:30.280,NaN,2025-01-03 13:17:01.100,NaN,0.0,0
9220,9220,9222,2025-01-03 12:18:56.297,2025-01-03 12:19:00.000,2025-01-03 12:53:42.357,NaN,2025-01-03 12:57:24.937,NaN,0.0,0
...,...,...,...,...,...,...,...,...,...,...
1724064,1724064,1724066,2025-12-30 19:57:46.610,2025-12-30 19:57:46.610,2025-12-30 20:08:17.320,NaN,2025-12-30 21:17:33.707,NaN,1500.0,0
1724356,1724356,1724358,2025-12-30 07:01:32.857,2025-12-31 07:00:00.000,2025-12-31 07:09:07.030,NaN,2025-12-31 08:46:29.960,NaN,0.0,0
1725043,1725043,1725045,2025-12-31 08:22:06.890,2025-12-31 08:22:00.000,2025-12-31 08:55:49.237,NaN,2025-12-31 09:47:50.150,NaN,0.0,0
1725500,1725500,1725502,2025-12-31 09:47:58.000,2025-12-31 09:49:00.000,2025-12-31 10:31:38.230,NaN,2025-12-31 13:07:16.880,NaN,0.0,0


### 승차일시가 없고 하차일시가 있는 행 제외

`승차일시`가 비어 있는데 `하차일시`가 있는 행은 탑승 시작 시각 없이 운행 종료 시각만 기록된 데이터이므로, 분석 기준상 논리적으로 맞지 않는 시간 데이터로 판단해 현재 노트북의 `df_raw`에서 제외한다. 원본 CSV 파일은 수정하지 않는다.

In [31]:
# EXCLUDE_MISSING_RIDE_AND_ALIGHT_EXISTS

# 승차일시가 없고 하차일시가 있는 행을 현재 df_raw에서 제외

missing_ride_and_alight_exists_indices = missing_ride_and_alight_exists.index

before_rows = len(df_raw)
df_raw = df_raw.drop(index=missing_ride_and_alight_exists_indices, errors='ignore')
after_rows = len(df_raw)

ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

remaining_missing_ride_and_alight_exists = df_raw[
    ride_dt.isna()
    & alight_dt.notna()
]

print(f'제외 전 행 수: {before_rows:,}')
print(f'제외 후 행 수: {after_rows:,}')
print(f'제외된 행 수: {before_rows - after_rows:,}')
print(f'남은 승차일시 없음 + 하차일시 있음 행 수: {len(remaining_missing_ride_and_alight_exists):,}')

df_raw.head()

제외 전 행 수: 1,729,435
제외 후 행 수: 1,728,744
제외된 행 수: 691
남은 승차일시 없음 + 하차일시 있음 행 수: 0


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


### 배차일시가 없고 승차일시가 있는 데이터 확인

`배차일시`가 비어 있는데 `승차일시`가 있는 행은 배차 시각 없이 실제 탑승 시각만 기록된 경우이므로, 배차 기록 누락 여부와 실제 운행 기록의 관계를 확인한다.

In [32]:
# CHECK_MISSING_DISPATCH_AND_RIDE_EXISTS

# 배차일시가 없고 승차일시가 있는 데이터 확인

dispatch_dt = pd.to_datetime(df_raw['배차일시'], errors='coerce')
ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')

missing_dispatch_and_ride_exists = df_raw[
    dispatch_dt.isna()
    & ride_dt.notna()
].copy()

missing_dispatch_and_ride_exists.insert(0, '원본인덱스', missing_dispatch_and_ride_exists.index)
missing_dispatch_and_ride_exists.insert(1, 'CSV행번호', missing_dispatch_and_ride_exists.index + 2)

print(f'배차일시가 없고 승차일시가 있는 행 수: {len(missing_dispatch_and_ride_exists):,}')

missing_dispatch_and_ride_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

배차일시가 없고 승차일시가 있는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 접수일시가 없고 이후 시간 기록이 있는 데이터 확인

`접수일시`가 비어 있는데 `배차일시`, `승차일시`, `하차일시`, `취소일시` 중 하나 이상이 있는 행은 최초 접수 시각 없이 후속 업무 시각만 기록된 경우이므로, 접수 기록 누락 여부와 분석 제외 필요성을 확인한다.

In [33]:
# CHECK_MISSING_REQUEST_AND_ANY_LATER_TIME_EXISTS

# 접수일시가 없고 배차/승차/하차/취소일시 중 하나 이상이 있는 데이터 확인

request_dt = pd.to_datetime(df_raw['접수일시'], errors='coerce')
dispatch_dt = pd.to_datetime(df_raw['배차일시'], errors='coerce')
ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

missing_request_and_any_later_time_exists = df_raw[
    request_dt.isna()
    & (
        dispatch_dt.notna()
        | ride_dt.notna()
        | alight_dt.notna()
        | cancel_dt.notna()
    )
].copy()

missing_request_and_any_later_time_exists.insert(0, '원본인덱스', missing_request_and_any_later_time_exists.index)
missing_request_and_any_later_time_exists.insert(1, 'CSV행번호', missing_request_and_any_later_time_exists.index + 2)

print(
    '접수일시가 없고 배차/승차/하차/취소일시 중 하나 이상이 있는 행 수: '
    f'{len(missing_request_and_any_later_time_exists):,}'
)

missing_request_and_any_later_time_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

접수일시가 없고 배차/승차/하차/취소일시 중 하나 이상이 있는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 승차거리가 0이고 하차일시가 있는 데이터 확인

`승차거리`가 0인데 `하차일시`가 있는 행은 운행 종료 시각은 기록되어 있지만 실제 이동 거리가 없는 경우이므로, 단거리 운행·기록 오류·취소성 운행 여부를 확인한다.

In [34]:
# CHECK_ZERO_DISTANCE_AND_ALIGHT_EXISTS

# 승차거리가 0이고 하차일시가 있는 데이터 확인

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

zero_distance_and_alight_exists = df_raw[
    (ride_distance == 0)
    & alight_dt.notna()
].copy()

zero_distance_and_alight_exists.insert(0, '원본인덱스', zero_distance_and_alight_exists.index)
zero_distance_and_alight_exists.insert(1, 'CSV행번호', zero_distance_and_alight_exists.index + 2)

print(f'승차거리가 0이고 하차일시가 있는 행 수: {len(zero_distance_and_alight_exists):,}')

zero_distance_and_alight_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

승차거리가 0이고 하차일시가 있는 행 수: 6,254


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리
39,39,41,2025-01-01 01:31:23.000,2025-01-01 01:32:00.000,2025-01-01 02:44:41.920,2025-01-01 03:05:54.970,2025-01-01 03:29:10.840,NaN,1500.0,0
495,495,497,2025-01-01 11:17:00.000,2025-01-01 11:17:00.000,2025-01-01 11:25:20.310,2025-01-01 11:46:42.393,2025-01-01 12:10:46.350,NaN,1500.0,0
644,644,646,2025-01-01 12:16:06.413,2025-01-01 12:16:00.000,2025-01-01 12:25:44.750,2025-01-01 13:12:32.760,2025-01-01 13:13:44.670,NaN,0.0,0
986,986,988,2025-01-01 15:00:20.000,2025-01-01 15:01:00.000,2025-01-01 15:08:00.240,2025-01-01 15:22:30.680,2025-01-01 15:26:03.837,NaN,1500.0,0
1166,1166,1168,2025-01-01 16:36:21.000,2025-01-01 16:37:00.000,2025-01-01 16:38:23.543,2025-01-01 16:45:11.700,2025-01-01 17:07:15.470,NaN,1500.0,0
...,...,...,...,...,...,...,...,...,...,...
1727059,1727059,1727061,2025-12-31 12:52:30.700,2025-12-31 12:52:30.700,2025-12-31 13:01:52.463,2025-12-31 13:23:47.307,2025-12-31 13:47:42.053,NaN,1500.0,0
1727078,1727078,1727080,2025-12-31 12:54:36.907,2025-12-31 12:55:00.000,2025-12-31 13:02:34.653,2025-12-31 13:34:32.243,2025-12-31 13:34:35.137,NaN,1500.0,0
1727079,1727079,1727081,2025-12-31 12:54:46.563,2025-12-31 12:55:00.000,2025-12-31 13:29:54.080,2025-12-31 13:30:00.033,2025-12-31 13:30:03.080,NaN,1500.0,0
1728033,1728033,1728035,2025-12-31 14:37:04.590,2025-12-31 14:37:00.000,2025-12-31 14:58:48.460,2025-12-31 15:22:12.990,2025-12-31 15:44:30.630,NaN,0.0,0


승차거리 0 + 요금 1500은 “요금 이상치”가 아니라, 기본요금으로 정상 처리됐을 가능성 => **“왜 승차거리가 0으로 찍혔나”**

In [35]:
# CHECK_ZERO_DISTANCE_AND_ALIGHT_EXISTS

# 승차거리가 0이고 하차일시가 있는 데이터 확인

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')
ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

zero_distance_and_alight_exists = df_raw[
    (ride_distance == 0)
    & alight_dt.notna()
].copy()

zero_distance_and_alight_exists.insert(0, '원본인덱스', zero_distance_and_alight_exists.index)
zero_distance_and_alight_exists.insert(1, 'CSV행번호', zero_distance_and_alight_exists.index + 2)

zero_distance_and_alight_exists['승차일시있음'] = ride_dt.loc[
    zero_distance_and_alight_exists.index
].notna()

zero_distance_and_alight_exists['요금상태'] = pd.NA
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index] == 1500,
    '요금상태'
] = '요금 1500'
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index] == 0,
    '요금상태'
] = '요금 0'
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index].isna(),
    '요금상태'
] = '요금 NaN'
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index].notna()
    & ~fare.loc[zero_distance_and_alight_exists.index].isin([0, 1500]),
    '요금상태'
] = '기타 요금'

print(f'승차거리가 0이고 하차일시가 있는 행 수: {len(zero_distance_and_alight_exists):,}')

print('[승차일시 있음 여부]')
display(
    zero_distance_and_alight_exists['승차일시있음']
    .value_counts(dropna=False)
    .reset_index(name='행 수')
)

print('[요금상태]')
display(
    zero_distance_and_alight_exists['요금상태']
    .value_counts(dropna=False)
    .reset_index(name='행 수')
)

print('[승차일시 있음 여부 x 요금상태]')
display(
    pd.crosstab(
        zero_distance_and_alight_exists['승차일시있음'],
        zero_distance_and_alight_exists['요금상태'],
        margins=True
    )
)

zero_distance_and_alight_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '승차일시있음',
        '요금상태',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

승차거리가 0이고 하차일시가 있는 행 수: 6,254
[승차일시 있음 여부]


,승차일시있음,행 수
0,True,6254


[요금상태]


,요금상태,행 수
0,요금 1500,4739
1,요금 0,810
2,기타 요금,705


[승차일시 있음 여부 x 요금상태]


요금상태,기타 요금,요금 0,요금 1500,All
승차일시있음,,,,
True,705,810,4739,6254
All,705,810,4739,6254


,원본인덱스,CSV행번호,승차일시있음,요금상태,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형
39,39,41,True,요금 1500,2025-01-01 01:31:23.000,2025-01-01 01:32:00.000,2025-01-01 02:44:41.920,2025-01-01 03:05:54.970,2025-01-01 03:29:10.840,NaN,1500.0,0,종로구,교남동,은평구,신사제1동,기타,특장차,지체
495,495,497,True,요금 1500,2025-01-01 11:17:00.000,2025-01-01 11:17:00.000,2025-01-01 11:25:20.310,2025-01-01 11:46:42.393,2025-01-01 12:10:46.350,NaN,1500.0,0,강남구,수서동,송파구,가락본동,치료,특장차,신장
644,644,646,True,요금 0,2025-01-01 12:16:06.413,2025-01-01 12:16:00.000,2025-01-01 12:25:44.750,2025-01-01 13:12:32.760,2025-01-01 13:13:44.670,NaN,0.0,0,용산구,청파동,종로구,숭인제2동,기타,특장차,지체
986,986,988,True,요금 1500,2025-01-01 15:00:20.000,2025-01-01 15:01:00.000,2025-01-01 15:08:00.240,2025-01-01 15:22:30.680,2025-01-01 15:26:03.837,NaN,1500.0,0,노원구,상계6.7동,노원구,상계9동,귀가,특장차,신장
1166,1166,1168,True,요금 1500,2025-01-01 16:36:21.000,2025-01-01 16:37:00.000,2025-01-01 16:38:23.543,2025-01-01 16:45:11.700,2025-01-01 17:07:15.470,NaN,1500.0,0,중랑구,면목제5동,중랑구,망우본동,귀가,임차택시,신장
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1727059,1727059,1727061,True,요금 1500,2025-12-31 12:52:30.700,2025-12-31 12:52:30.700,2025-12-31 13:01:52.463,2025-12-31 13:23:47.307,2025-12-31 13:47:42.053,NaN,1500.0,0,노원구,상계1동,노원구,상계6.7동,기타,특장차,뇌병
1727078,1727078,1727080,True,요금 1500,2025-12-31 12:54:36.907,2025-12-31 12:55:00.000,2025-12-31 13:02:34.653,2025-12-31 13:34:32.243,2025-12-31 13:34:35.137,NaN,1500.0,0,성동구,금호4가동,성동구,성수1가제1동,기타,임차택시,뇌병
1727079,1727079,1727081,True,요금 1500,2025-12-31 12:54:46.563,2025-12-31 12:55:00.000,2025-12-31 13:29:54.080,2025-12-31 13:30:00.033,2025-12-31 13:30:03.080,NaN,1500.0,0,노원구,중계2.3동,노원구,상계6.7동,기타,특장차,뇌병
1728033,1728033,1728035,True,요금 0,2025-12-31 14:37:04.590,2025-12-31 14:37:00.000,2025-12-31 14:58:48.460,2025-12-31 15:22:12.990,2025-12-31 15:44:30.630,NaN,0.0,0,성북구,월곡제2동,성북구,정릉제4동,기타,특장차,뇌병


### 승차거리 0, 하차일시 있음 행 중 기타 요금 확인

`승차거리`가 0이고 `하차일시`가 있는 행 중 요금이 0원이나 기본요금 1500원이 아닌 경우를 별도로 확인한다. 기타 요금 값의 고유값과 빈도를 함께 살펴보아 특정 요금 패턴이 반복되는지, 거리 기록 누락이나 요금 입력 오류 가능성이 있는지 점검한다.

In [36]:
# CHECK_ZERO_DISTANCE_AND_OTHER_FARE

# 승차거리 0, 하차일시 있음 행 중 기타 요금 데이터 확인

zero_distance_and_other_fare = zero_distance_and_alight_exists[
    zero_distance_and_alight_exists['요금상태'] == '기타 요금'
].copy()

other_fare_numeric = pd.to_numeric(
    zero_distance_and_other_fare['요금'],
    errors='coerce'
)

print(f'승차거리 0 + 하차일시 있음 + 기타 요금 행 수: {len(zero_distance_and_other_fare):,}')
print(f'기타 요금 unique 값 수: {other_fare_numeric.nunique(dropna=False):,}')

print('[기타 요금 unique 값]')
display(
    pd.DataFrame({
        '요금': sorted(other_fare_numeric.dropna().unique())
    })
)

print('[기타 요금 값 빈도]')
display(
    other_fare_numeric
    .value_counts(dropna=False)
    .sort_index()
    .reset_index(name='행 수')
)

zero_distance_and_other_fare[
    [
        '원본인덱스',
        'CSV행번호',
        '승차일시있음',
        '요금상태',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

승차거리 0 + 하차일시 있음 + 기타 요금 행 수: 705
기타 요금 unique 값 수: 1
[기타 요금 unique 값]


,요금
0,45000.0


[기타 요금 값 빈도]


,요금,행 수
0,45000.0,705


,원본인덱스,CSV행번호,승차일시있음,요금상태,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형
2812,2812,2814,True,기타 요금,2025-01-01 10:34:56.000,2025-01-02 10:00:00.000,2025-01-02 10:09:47.807,2025-01-02 10:17:32.930,2025-01-02 11:07:29.380,NaN,45000.0,0,강남구,세곡동,강남구,일원2동,예약재활,임차택시,지체
2840,2840,2842,True,기타 요금,2025-01-02 09:59:08.000,2025-01-02 10:00:00.000,2025-01-02 10:03:44.083,2025-01-02 10:29:19.383,2025-01-02 10:44:06.733,NaN,45000.0,0,양천구,신정3동,양천구,신월7동,귀가,임차택시,뇌병
3613,3613,3615,True,기타 요금,2025-01-02 11:50:52.060,2025-01-02 11:50:52.060,2025-01-02 11:54:10.157,2025-01-02 15:49:04.420,2025-01-02 15:49:06.873,NaN,45000.0,0,구로구,개봉제2동,구로구,수궁동,기타,임차택시,뇌병
6359,6359,6361,True,기타 요금,2025-01-02 17:34:16.000,2025-01-02 17:35:00.000,2025-01-02 17:35:42.937,2025-01-02 17:45:58.193,2025-01-02 17:57:16.340,NaN,45000.0,0,노원구,공릉2동,노원구,공릉1.3동,기타,임차택시,뇌병
7046,7046,7048,True,기타 요금,2025-01-03 07:19:29.477,2025-01-03 07:19:29.477,2025-01-03 07:42:15.947,2025-01-03 08:35:29.447,2025-01-03 08:36:41.447,NaN,45000.0,0,관악구,청림동,동작구,신대방제2동,기타,임차택시,지체
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427179,1427179,1427181,True,기타 요금,2025-10-30 10:02:08.043,2025-10-31 10:00:00.000,2025-10-31 10:01:50.880,2025-10-31 10:11:24.133,2025-10-31 10:11:30.057,NaN,45000.0,0,마포구,상암동,서대문구,북가좌제2동,예약기타,임차택시,지체
1440781,1440781,1440783,True,기타 요금,2025-11-03 15:00:59.000,2025-11-03 15:01:00.000,2025-11-03 15:12:31.013,2025-11-03 15:22:13.627,2025-11-03 15:37:05.483,NaN,45000.0,0,노원구,공릉1.3동,노원구,월계1동,기타,임차택시,신장
1443787,1443787,1443789,True,기타 요금,2025-11-03 10:00:40.000,2025-11-04 10:00:00.000,2025-11-04 10:02:59.743,2025-11-04 10:13:27.367,2025-11-04 10:19:42.137,NaN,45000.0,0,도봉구,도봉제1동,도봉구,방학제1동,예약기타,임차택시,지체
1444877,1444877,1444879,True,기타 요금,2025-11-04 12:05:48.630,2025-11-04 12:05:48.630,2025-11-04 12:08:20.713,2025-11-04 12:28:39.490,2025-11-04 13:12:14.277,NaN,45000.0,0,서초구,양재2동,종로구,이화동,기타,임차택시,지적


### 승차거리 0, 하차일시 있음, 요금 45000원 행 제외

`승차거리`가 0이고 `하차일시`가 있는데 `요금`이 45000원인 행은 운행 종료 기록과 고액 요금이 함께 있으나 이동 거리가 없는 데이터이므로, 거리 기록과 요금 기록이 논리적으로 맞지 않는 행으로 판단해 현재 노트북의 `df_raw`에서 제외한다. 원본 CSV 파일은 수정하지 않는다.

In [37]:
# EXCLUDE_ZERO_DISTANCE_ALIGHT_EXISTS_FARE_45000

# 승차거리 0, 하차일시 있음, 요금 45000원 행을 현재 df_raw에서 제외

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')

zero_distance_alight_exists_fare_45000_indices = df_raw[
    (ride_distance == 0)
    & alight_dt.notna()
    & (fare == 45000)
].index

before_rows = len(df_raw)
df_raw = df_raw.drop(index=zero_distance_alight_exists_fare_45000_indices, errors='ignore')
after_rows = len(df_raw)

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')

remaining_zero_distance_alight_exists_fare_45000 = df_raw[
    (ride_distance == 0)
    & alight_dt.notna()
    & (fare == 45000)
]

print(f'제외 전 행 수: {before_rows:,}')
print(f'제외 후 행 수: {after_rows:,}')
print(f'제외된 행 수: {before_rows - after_rows:,}')
print(
    '남은 승차거리 0 + 하차일시 있음 + 요금 45000원 행 수: '
    f'{len(remaining_zero_distance_alight_exists_fare_45000):,}'
)

df_raw.head()

제외 전 행 수: 1,728,744
제외 후 행 수: 1,728,039
제외된 행 수: 705
남은 승차거리 0 + 하차일시 있음 + 요금 45000원 행 수: 0


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


### 승차거리가 0이고 하차일시가 있는 데이터 재확인

요금 45000원 이상 행을 제외한 뒤에도 `승차거리`가 0이고 `하차일시`가 있는 행이 남아 있는지 다시 확인하고, 승차일시 존재 여부와 요금 상태별 분포를 점검한다.

In [38]:
# RECHECK_ZERO_DISTANCE_AND_ALIGHT_EXISTS

# 승차거리 0, 하차일시 있음 데이터 재확인

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')
ride_dt = pd.to_datetime(df_raw['승차일시'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

zero_distance_and_alight_exists = df_raw[
    (ride_distance == 0)
    & alight_dt.notna()
].copy()

zero_distance_and_alight_exists.insert(0, '원본인덱스', zero_distance_and_alight_exists.index)
zero_distance_and_alight_exists.insert(1, 'CSV행번호', zero_distance_and_alight_exists.index + 2)

zero_distance_and_alight_exists['승차일시있음'] = ride_dt.loc[
    zero_distance_and_alight_exists.index
].notna()

zero_distance_and_alight_exists['요금상태'] = pd.NA
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index] == 1500,
    '요금상태'
] = '요금 1500'
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index] == 0,
    '요금상태'
] = '요금 0'
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index].isna(),
    '요금상태'
] = '요금 NaN'
zero_distance_and_alight_exists.loc[
    fare.loc[zero_distance_and_alight_exists.index].notna()
    & ~fare.loc[zero_distance_and_alight_exists.index].isin([0, 1500]),
    '요금상태'
] = '기타 요금'

print(f'승차거리가 0이고 하차일시가 있는 행 수: {len(zero_distance_and_alight_exists):,}')

print('[승차일시 있음 여부]')
display(
    zero_distance_and_alight_exists['승차일시있음']
    .value_counts(dropna=False)
    .reset_index(name='행 수')
)

print('[요금상태]')
display(
    zero_distance_and_alight_exists['요금상태']
    .value_counts(dropna=False)
    .reset_index(name='행 수')
)

print('[승차일시 있음 여부 x 요금상태]')
display(
    pd.crosstab(
        zero_distance_and_alight_exists['승차일시있음'],
        zero_distance_and_alight_exists['요금상태'],
        margins=True
    )
)

zero_distance_and_alight_exists[
    [
        '원본인덱스',
        'CSV행번호',
        '승차일시있음',
        '요금상태',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

승차거리가 0이고 하차일시가 있는 행 수: 5,549
[승차일시 있음 여부]


,승차일시있음,행 수
0,True,5549


[요금상태]


,요금상태,행 수
0,요금 1500,4739
1,요금 0,810


[승차일시 있음 여부 x 요금상태]


요금상태,요금 0,요금 1500,All
승차일시있음,,,
True,810,4739,5549
All,810,4739,5549


,원본인덱스,CSV행번호,승차일시있음,요금상태,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형
39,39,41,True,요금 1500,2025-01-01 01:31:23.000,2025-01-01 01:32:00.000,2025-01-01 02:44:41.920,2025-01-01 03:05:54.970,2025-01-01 03:29:10.840,NaN,1500.0,0,종로구,교남동,은평구,신사제1동,기타,특장차,지체
495,495,497,True,요금 1500,2025-01-01 11:17:00.000,2025-01-01 11:17:00.000,2025-01-01 11:25:20.310,2025-01-01 11:46:42.393,2025-01-01 12:10:46.350,NaN,1500.0,0,강남구,수서동,송파구,가락본동,치료,특장차,신장
644,644,646,True,요금 0,2025-01-01 12:16:06.413,2025-01-01 12:16:00.000,2025-01-01 12:25:44.750,2025-01-01 13:12:32.760,2025-01-01 13:13:44.670,NaN,0.0,0,용산구,청파동,종로구,숭인제2동,기타,특장차,지체
986,986,988,True,요금 1500,2025-01-01 15:00:20.000,2025-01-01 15:01:00.000,2025-01-01 15:08:00.240,2025-01-01 15:22:30.680,2025-01-01 15:26:03.837,NaN,1500.0,0,노원구,상계6.7동,노원구,상계9동,귀가,특장차,신장
1166,1166,1168,True,요금 1500,2025-01-01 16:36:21.000,2025-01-01 16:37:00.000,2025-01-01 16:38:23.543,2025-01-01 16:45:11.700,2025-01-01 17:07:15.470,NaN,1500.0,0,중랑구,면목제5동,중랑구,망우본동,귀가,임차택시,신장
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1727059,1727059,1727061,True,요금 1500,2025-12-31 12:52:30.700,2025-12-31 12:52:30.700,2025-12-31 13:01:52.463,2025-12-31 13:23:47.307,2025-12-31 13:47:42.053,NaN,1500.0,0,노원구,상계1동,노원구,상계6.7동,기타,특장차,뇌병
1727078,1727078,1727080,True,요금 1500,2025-12-31 12:54:36.907,2025-12-31 12:55:00.000,2025-12-31 13:02:34.653,2025-12-31 13:34:32.243,2025-12-31 13:34:35.137,NaN,1500.0,0,성동구,금호4가동,성동구,성수1가제1동,기타,임차택시,뇌병
1727079,1727079,1727081,True,요금 1500,2025-12-31 12:54:46.563,2025-12-31 12:55:00.000,2025-12-31 13:29:54.080,2025-12-31 13:30:00.033,2025-12-31 13:30:03.080,NaN,1500.0,0,노원구,중계2.3동,노원구,상계6.7동,기타,특장차,뇌병
1728033,1728033,1728035,True,요금 0,2025-12-31 14:37:04.590,2025-12-31 14:37:00.000,2025-12-31 14:58:48.460,2025-12-31 15:22:12.990,2025-12-31 15:44:30.630,NaN,0.0,0,성북구,월곡제2동,성북구,정릉제4동,기타,특장차,뇌병


### 승차거리 0, 하차일시 있음 행 중 요금 1500원 데이터 확인

`승차거리`가 0이고 `하차일시`가 있는 행 중 요금이 기본요금 1500원으로 기록된 데이터를 별도로 확인한다. 기본요금 처리된 단거리 운행인지, 거리 기록이 누락된 운행인지 판단하기 위해 승차일시와 출발·목적 지역 정보를 함께 살펴본다.

In [39]:
# CHECK_ZERO_DISTANCE_ALIGHT_EXISTS_FARE_1500

# 승차거리 0, 하차일시 있음 행 중 요금 1500원 데이터 확인

zero_distance_alight_exists_fare_1500 = zero_distance_and_alight_exists[
    zero_distance_and_alight_exists['요금상태'] == '요금 1500'
].copy()

print(
    '승차거리 0 + 하차일시 있음 + 요금 1500원 행 수: '
    f'{len(zero_distance_alight_exists_fare_1500):,}'
)

zero_distance_alight_exists_fare_1500[
    [
        '원본인덱스',
        'CSV행번호',
        '승차일시있음',
        '요금상태',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

승차거리 0 + 하차일시 있음 + 요금 1500원 행 수: 4,739


,원본인덱스,CSV행번호,승차일시있음,요금상태,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형
39,39,41,True,요금 1500,2025-01-01 01:31:23.000,2025-01-01 01:32:00.000,2025-01-01 02:44:41.920,2025-01-01 03:05:54.970,2025-01-01 03:29:10.840,NaN,1500.0,0,종로구,교남동,은평구,신사제1동,기타,특장차,지체
495,495,497,True,요금 1500,2025-01-01 11:17:00.000,2025-01-01 11:17:00.000,2025-01-01 11:25:20.310,2025-01-01 11:46:42.393,2025-01-01 12:10:46.350,NaN,1500.0,0,강남구,수서동,송파구,가락본동,치료,특장차,신장
986,986,988,True,요금 1500,2025-01-01 15:00:20.000,2025-01-01 15:01:00.000,2025-01-01 15:08:00.240,2025-01-01 15:22:30.680,2025-01-01 15:26:03.837,NaN,1500.0,0,노원구,상계6.7동,노원구,상계9동,귀가,특장차,신장
1166,1166,1168,True,요금 1500,2025-01-01 16:36:21.000,2025-01-01 16:37:00.000,2025-01-01 16:38:23.543,2025-01-01 16:45:11.700,2025-01-01 17:07:15.470,NaN,1500.0,0,중랑구,면목제5동,중랑구,망우본동,귀가,임차택시,신장
1447,1447,1449,True,요금 1500,2025-01-02 01:45:00.613,2025-01-02 03:50:00.000,2025-01-02 03:43:32.340,2025-01-02 04:01:21.077,2025-01-02 04:34:23.270,NaN,1500.0,0,중랑구,면목제5동,동대문구,휘경제2동,치료,특장차,시각
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1726972,1726972,1726974,True,요금 1500,2025-12-31 12:42:26.783,2025-12-31 12:42:26.783,2025-12-31 12:51:19.387,2025-12-31 13:34:38.900,2025-12-31 13:34:50.790,NaN,1500.0,0,강동구,강일동,강동구,둔촌제2동,기타,특장차,뇌병
1727059,1727059,1727061,True,요금 1500,2025-12-31 12:52:30.700,2025-12-31 12:52:30.700,2025-12-31 13:01:52.463,2025-12-31 13:23:47.307,2025-12-31 13:47:42.053,NaN,1500.0,0,노원구,상계1동,노원구,상계6.7동,기타,특장차,뇌병
1727078,1727078,1727080,True,요금 1500,2025-12-31 12:54:36.907,2025-12-31 12:55:00.000,2025-12-31 13:02:34.653,2025-12-31 13:34:32.243,2025-12-31 13:34:35.137,NaN,1500.0,0,성동구,금호4가동,성동구,성수1가제1동,기타,임차택시,뇌병
1727079,1727079,1727081,True,요금 1500,2025-12-31 12:54:46.563,2025-12-31 12:55:00.000,2025-12-31 13:29:54.080,2025-12-31 13:30:00.033,2025-12-31 13:30:03.080,NaN,1500.0,0,노원구,중계2.3동,노원구,상계6.7동,기타,특장차,뇌병


### 승차거리 0, 하차일시 있음, 요금 1500원 행 중 출발구와 목적구가 다른 데이터 확인

`승차거리`가 0이고 `하차일시`가 있으며 요금이 기본요금 1500원인 행 중 `출발구`와 `목적구`가 다른 경우를 확인한다. 서로 다른 구 간 이동인데 승차거리가 0으로 기록된 행은 거리 기록 누락 가능성이 있으므로 별도로 살펴본다.

In [40]:
# CHECK_ZERO_DISTANCE_ALIGHT_EXISTS_FARE_1500_DIFFERENT_DISTRICT

# 승차거리 0, 하차일시 있음, 요금 1500원 행 중 출발구와 목적구가 다른 데이터 확인

zero_distance_alight_exists_fare_1500_different_district = zero_distance_alight_exists_fare_1500[
    zero_distance_alight_exists_fare_1500['출발구'].notna()
    & zero_distance_alight_exists_fare_1500['목적구'].notna()
    & (zero_distance_alight_exists_fare_1500['출발구'] != zero_distance_alight_exists_fare_1500['목적구'])
].copy()

print(
    '승차거리 0 + 하차일시 있음 + 요금 1500원 + 출발구/목적구 다름 행 수: '
    f'{len(zero_distance_alight_exists_fare_1500_different_district):,}'
)

zero_distance_alight_exists_fare_1500_different_district[
    [
        '원본인덱스',
        'CSV행번호',
        '승차일시있음',
        '요금상태',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

승차거리 0 + 하차일시 있음 + 요금 1500원 + 출발구/목적구 다름 행 수: 2,610


,원본인덱스,CSV행번호,승차일시있음,요금상태,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형
39,39,41,True,요금 1500,2025-01-01 01:31:23.000,2025-01-01 01:32:00.000,2025-01-01 02:44:41.920,2025-01-01 03:05:54.970,2025-01-01 03:29:10.840,NaN,1500.0,0,종로구,교남동,은평구,신사제1동,기타,특장차,지체
495,495,497,True,요금 1500,2025-01-01 11:17:00.000,2025-01-01 11:17:00.000,2025-01-01 11:25:20.310,2025-01-01 11:46:42.393,2025-01-01 12:10:46.350,NaN,1500.0,0,강남구,수서동,송파구,가락본동,치료,특장차,신장
1447,1447,1449,True,요금 1500,2025-01-02 01:45:00.613,2025-01-02 03:50:00.000,2025-01-02 03:43:32.340,2025-01-02 04:01:21.077,2025-01-02 04:34:23.270,NaN,1500.0,0,중랑구,면목제5동,동대문구,휘경제2동,치료,특장차,시각
1479,1479,1481,True,요금 1500,2025-01-02 03:15:00.093,2025-01-02 05:20:00.000,2025-01-02 04:51:53.643,2025-01-02 05:15:48.437,2025-01-02 05:55:10.237,NaN,1500.0,0,강북구,송천동,도봉구,쌍문제3동,치료,특장차,시각
2607,2607,2609,True,요금 1500,2025-01-02 09:28:57.233,2025-01-02 09:28:57.233,2025-01-02 09:34:38.610,2025-01-02 10:11:30.730,2025-01-02 10:11:33.620,NaN,1500.0,0,강서구,등촌제1동,양천구,신정6동,기타,특장차,뇌병
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1720983,1720983,1720985,True,요금 1500,2025-12-30 11:58:22.957,2025-12-30 11:58:22.957,2025-12-30 12:10:18.673,2025-12-30 12:22:28.197,2025-12-30 13:03:53.940,NaN,1500.0,0,은평구,진관동,서대문구,신촌동,기타,특장차,뇌병
1721194,1721194,1721196,True,요금 1500,2025-12-30 12:16:52.000,2025-12-30 12:17:00.000,2025-12-30 12:33:47.590,2025-12-30 13:24:07.167,2025-12-30 13:24:28.230,NaN,1500.0,0,영등포구,영등포본동,금천구,독산제3동,기타,특장차,지체
1721628,1721628,1721630,True,요금 1500,2025-12-30 13:04:42.400,2025-12-30 13:04:42.400,2025-12-30 13:15:47.993,2025-12-30 13:45:48.813,2025-12-30 13:49:17.143,NaN,1500.0,0,강북구,번제3동,성북구,길음제2동,기타,특장차,뇌병
1722306,1722306,1722308,True,요금 1500,2025-12-30 14:13:25.033,2025-12-30 14:13:00.000,2025-12-30 15:54:45.783,2025-12-30 16:31:16.780,2025-12-30 17:35:14.277,NaN,1500.0,0,노원구,상계1동,마포구,서교동,기타,특장차,지적


### 승차거리 0, 하차일시 있음 행 제외

`승차거리`가 0이고 `하차일시`가 있는 행은 운행 종료 기록이 있음에도 이동 거리가 0으로 기록된 데이터이다. 이 행들을 확인한 결과 요금은 기본요금 1500원 또는 0원으로만 나타났지만, 실제로는 운행시간이 매우 짧은데 요금이 0원이거나 출발지와 목적지가 달라 5km 초과 운행 가능성이 있는데도 기본요금 수준으로 기록된 사례가 포함되어 있었다. 따라서 거리와 요금 기록의 신뢰성이 낮다고 판단해 현재 노트북의 `df_raw`에서 제외한다. 원본 CSV 파일은 수정하지 않는다.

In [41]:
# EXCLUDE_ZERO_DISTANCE_AND_ALIGHT_EXISTS

# 승차거리 0, 하차일시 있음 행을 현재 df_raw에서 제외

zero_distance_and_alight_exists_indices = zero_distance_and_alight_exists.index

before_rows = len(df_raw)
df_raw = df_raw.drop(index=zero_distance_and_alight_exists_indices, errors='ignore')
after_rows = len(df_raw)

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')

remaining_zero_distance_and_alight_exists = df_raw[
    (ride_distance == 0)
    & alight_dt.notna()
]

print(f'제외 전 행 수: {before_rows:,}')
print(f'제외 후 행 수: {after_rows:,}')
print(f'제외된 행 수: {before_rows - after_rows:,}')
print(
    '남은 승차거리 0 + 하차일시 있음 행 수: '
    f'{len(remaining_zero_distance_and_alight_exists):,}'
)

df_raw.head()

제외 전 행 수: 1,728,039
제외 후 행 수: 1,722,490
제외된 행 수: 5,549
남은 승차거리 0 + 하차일시 있음 행 수: 0


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


### 하차일시가 있고 요금이 없는 데이터 확인

`하차일시`가 있는데 `요금`이 비어 있는 행은 운행 종료 시각은 기록되어 있지만 요금 정보가 누락된 경우이므로, 요금 기반 분석에서 제외 또는 보정이 필요한지 확인한다.

In [42]:
# CHECK_ALIGHT_AND_MISSING_FARE

# 하차일시가 있고 요금이 없는 데이터 확인

alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')

alight_and_missing_fare = df_raw[
    alight_dt.notna()
    & fare.isna()
].copy()

alight_and_missing_fare.insert(0, '원본인덱스', alight_and_missing_fare.index)
alight_and_missing_fare.insert(1, 'CSV행번호', alight_and_missing_fare.index + 2)

print(f'하차일시가 있고 요금이 없는 행 수: {len(alight_and_missing_fare):,}')

alight_and_missing_fare[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

하차일시가 있고 요금이 없는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형


### 승차거리가 0보다 크고 요금이 없는 데이터 확인

`승차거리`가 0보다 큰데 `요금`이 비어 있는 행은 실제 이동 거리는 기록되어 있지만 요금 기록이 누락된 경우이므로, 요금 결측 여부와 분석 처리 필요성을 확인한다.

In [43]:
# CHECK_POSITIVE_DISTANCE_AND_MISSING_FARE

# 승차거리가 0보다 크고 요금이 없는 데이터 확인

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')

positive_distance_and_missing_fare = df_raw[
    (ride_distance > 0)
    & fare.isna()
].copy()

positive_distance_and_missing_fare.insert(0, '원본인덱스', positive_distance_and_missing_fare.index)
positive_distance_and_missing_fare.insert(1, 'CSV행번호', positive_distance_and_missing_fare.index + 2)

print(f'승차거리가 0보다 크고 요금이 없는 행 수: {len(positive_distance_and_missing_fare):,}')

positive_distance_and_missing_fare[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

승차거리가 0보다 크고 요금이 없는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 요금이 0보다 크고 승차거리가 없거나 0인 데이터 확인

`요금`이 0보다 큰데 `승차거리`가 비어 있거나 0인 행은 요금은 발생했지만 실제 이동 거리가 없거나 누락된 것으로 기록된 경우이므로, 거리 기록 오류나 특수 과금 여부를 확인한다.

In [44]:
# CHECK_POSITIVE_FARE_AND_MISSING_OR_ZERO_DISTANCE

# 요금이 0보다 크고 승차거리가 없거나 0인 데이터 확인

fare = pd.to_numeric(df_raw['요금'], errors='coerce')
ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')

positive_fare_and_missing_or_zero_distance = df_raw[
    (fare > 0)
    & (ride_distance.isna() | (ride_distance == 0))
].copy()

positive_fare_and_missing_or_zero_distance.insert(0, '원본인덱스', positive_fare_and_missing_or_zero_distance.index)
positive_fare_and_missing_or_zero_distance.insert(1, 'CSV행번호', positive_fare_and_missing_or_zero_distance.index + 2)

print(f'요금이 0보다 크고 승차거리가 없거나 0인 행 수: {len(positive_fare_and_missing_or_zero_distance):,}')

positive_fare_and_missing_or_zero_distance[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

요금이 0보다 크고 승차거리가 없거나 0인 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 승차거리와 요금이 0이고 하차일시가 있으며 취소가 아닌 데이터 확인

`승차거리`와 `요금`이 모두 0이고 `하차일시`가 있으며 `취소일시`가 없는 행은 운행 종료 기록은 있지만 거리와 요금이 발생하지 않은 경우이므로, 무료·감면 운행인지 또는 거리·요금 기록 오류인지 확인한다.

In [45]:
# CHECK_ZERO_DISTANCE_ZERO_FARE_ALIGHT_EXISTS_NOT_CANCELED

# 승차거리 0, 요금 0, 하차일시 있음, 취소일시 없음 데이터 확인

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')
fare = pd.to_numeric(df_raw['요금'], errors='coerce')
alight_dt = pd.to_datetime(df_raw['하차일시'], errors='coerce')
cancel_dt = pd.to_datetime(df_raw['취소일시'], errors='coerce')

zero_distance_zero_fare_alight_exists_not_canceled = df_raw[
    (ride_distance == 0)
    & (fare == 0)
    & alight_dt.notna()
    & cancel_dt.isna()
].copy()

zero_distance_zero_fare_alight_exists_not_canceled.insert(
    0,
    '원본인덱스',
    zero_distance_zero_fare_alight_exists_not_canceled.index
)
zero_distance_zero_fare_alight_exists_not_canceled.insert(
    1,
    'CSV행번호',
    zero_distance_zero_fare_alight_exists_not_canceled.index + 2
)

print(
    '승차거리와 요금이 0이고 하차일시가 있으며 취소일시가 없는 행 수: '
    f'{len(zero_distance_zero_fare_alight_exists_not_canceled):,}'
)

zero_distance_zero_fare_alight_exists_not_canceled[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

승차거리와 요금이 0이고 하차일시가 있으며 취소일시가 없는 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형


In [46]:
# CHECK_DELETED_ROWS_ON_APRIL_20

# 지금까지 df_raw에서 제거된 행 중 4월 20일 데이터가 있는지 확인
# 4월 20일 장애인의 날 0시부터 24시까지 서울장애인콜택시를 무료 운행

original_df = pd.read_csv(input_path)

deleted_indices = original_df.index.difference(df_raw.index)
deleted_rows = original_df.loc[deleted_indices].copy()

deleted_rows.insert(0, '원본인덱스', deleted_rows.index)
deleted_rows.insert(1, 'CSV행번호', deleted_rows.index + 2)

time_columns = [
    '접수일시',
    '예정일시',
    '배차일시',
    '승차일시',
    '하차일시',
    '취소일시'
]

april_20_mask = False

for col in time_columns:
    dt = pd.to_datetime(deleted_rows[col], errors='coerce')
    april_20_mask = april_20_mask | ((dt.dt.month == 4) & (dt.dt.day == 20))

deleted_rows_on_april_20 = deleted_rows[april_20_mask].copy()

print(f'지금까지 제거된 행 수: {len(deleted_rows):,}')
print(f'제거된 행 중 4월 20일 관련 행 수: {len(deleted_rows_on_april_20):,}')

deleted_rows_on_april_20[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '차량구분',
        '장애유형'
    ]
]

지금까지 제거된 행 수: 6,986
제거된 행 중 4월 20일 관련 행 수: 81


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리,출발구,출발동,목적구,목적동,이용목적,차량구분,장애유형
490988,490988,490990,2025-04-19 20:42:32.120,2025-04-19 20:42:32.120,2025-04-19 21:02:08.300,2025-04-19 21:13:11.837,2025-04-20 00:03:17.860,NaN,0.0,0,노원구,중계2.3동,도봉구,도봉제1동,기타,특장차,지체
491001,491001,491003,2025-04-19 21:21:17.000,2025-04-19 21:22:00.000,2025-04-19 21:57:20.710,2025-04-20 00:02:41.687,2025-04-20 00:02:44.607,NaN,1500.0,0,강동구,천호제1동,성남시분당구,판교동,귀가,특장차,뇌병
491112,491112,491114,2025-04-19 11:03:18.000,2025-04-20 07:00:00.000,2025-04-20 07:08:09.170,2025-04-20 07:21:27.707,2025-04-20 07:38:13.513,NaN,0.0,0,노원구,상계6.7동,노원구,월계1동,예약기타,특장차,지체
491134,491134,491136,2025-04-20 06:21:55.000,2025-04-20 07:00:00.000,2025-04-20 07:03:09.150,2025-04-20 07:35:15.857,2025-04-20 08:06:29.103,NaN,1500.0,0,강남구,세곡동,성남시분당구,서현1동,기타,특장차,신장
491160,491160,491162,2025-04-20 07:14:02.000,2025-04-20 07:15:00.000,2025-04-20 08:09:08.590,2025-04-20 08:23:27.083,2025-04-20 08:58:50.080,NaN,1500.0,0,노원구,하계1동,종로구,종로1.2.3.4가동,기타,특장차,뇌병
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493740,493740,493742,2025-04-20 19:00:30.773,2025-04-20 19:00:30.773,2025-04-20 21:18:12.797,2025-04-20 21:43:55.700,2025-04-20 22:35:40.617,NaN,1500.0,0,종로구,이화동,고양시일산동구,중산동,기타,특장차,뇌병
493851,493851,493853,2025-04-20 20:01:35.897,2025-04-20 20:02:00.000,2025-04-20 23:32:17.830,NaN,2025-04-21 00:22:52.863,NaN,0.0,0,강동구,암사제1동,강동구,천호제1동,기타,특장차,지체
493921,493921,493923,2025-04-20 22:40:24.000,2025-04-20 22:41:00.000,2025-04-21 00:18:19.783,NaN,2025-04-21 01:10:34.303,NaN,0.0,0,동대문구,용두동,노원구,상계1동,귀가,특장차,뇌병
493925,493925,493927,2025-04-20 22:50:40.657,2025-04-20 22:51:00.000,2025-04-20 23:37:58.630,2025-04-20 23:57:35.067,2025-04-21 00:49:04.550,NaN,0.0,0,강서구,발산제1동,서초구,내곡동,기타,특장차,뇌병


### 요금이 음수인 데이터 확인

`요금`이 0보다 작은 행은 일반적인 요금 체계에서는 발생하기 어려운 값이므로, 환불·보정 입력 여부 또는 기록 오류 가능성을 확인한다.

In [47]:
# CHECK_NEGATIVE_FARE

# 요금이 음수인 데이터 확인

fare = pd.to_numeric(df_raw['요금'], errors='coerce')

negative_fare = df_raw[
    fare < 0
].copy()

negative_fare.insert(0, '원본인덱스', negative_fare.index)
negative_fare.insert(1, 'CSV행번호', negative_fare.index + 2)

print(f'요금이 음수인 행 수: {len(negative_fare):,}')

negative_fare[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

요금이 음수인 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 승차거리가 음수인 데이터 확인

`승차거리`가 0보다 작은 행은 일반적인 운행 기록에서는 발생하기 어려운 값이므로, 거리 보정 입력 여부 또는 기록 오류 가능성을 확인한다.

In [48]:
# CHECK_NEGATIVE_DISTANCE

# 승차거리가 음수인 데이터 확인

ride_distance = pd.to_numeric(df_raw['승차거리'], errors='coerce')

negative_distance = df_raw[
    ride_distance < 0
].copy()

negative_distance.insert(0, '원본인덱스', negative_distance.index)
negative_distance.insert(1, 'CSV행번호', negative_distance.index + 2)

print(f'승차거리가 음수인 행 수: {len(negative_distance):,}')

negative_distance[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '요금',
        '승차거리'
    ]
]

승차거리가 음수인 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,요금,승차거리


### 주요 컬럼 기준 전체 중복 데이터 확인

접수·예정·배차·승차·하차·취소 시각과 요금, 승차거리 등 주요 분석 컬럼이 모두 같은 행은 동일 운행 기록이 중복 입력되었을 가능성이 있으므로 확인한다.

In [49]:
# CHECK_DUPLICATED_MAIN_COLUMNS

# 주요 컬럼 기준 전체 중복 데이터 확인

main_columns = [
    '접수일시',
    '예정일시',
    '배차일시',
    '승차일시',
    '하차일시',
    '취소일시',
    '요금',
    '승차거리'
]

duplicated_main_columns = df_raw[
    df_raw.duplicated(subset=main_columns, keep=False)
].copy()

duplicated_main_columns.insert(0, '원본인덱스', duplicated_main_columns.index)
duplicated_main_columns.insert(1, 'CSV행번호', duplicated_main_columns.index + 2)

print(f'주요 컬럼 기준 전체 중복 행 수: {len(duplicated_main_columns):,}')

duplicated_main_columns.sort_values(main_columns).head(10)

주요 컬럼 기준 전체 중복 행 수: 130


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
5066,5066,5068,2025-01-02 12:25:00.703,2025-01-02 14:30:00.000,NaN,NaN,NaN,2025-01-02 12:26:43.000,강서구,가양제2동,강서구,발산제1동,치료,NaN,0,특장차,자폐
5067,5067,5069,2025-01-02 12:25:00.703,2025-01-02 14:30:00.000,NaN,NaN,NaN,2025-01-02 12:26:43.000,도봉구,창제1동,도봉구,창제2동,재활,NaN,0,특장차,뇌병
30638,30638,30640,2025-01-08 11:05:00.190,2025-01-08 13:10:00.000,NaN,NaN,NaN,2025-01-08 11:05:44.000,마포구,성산제2동,마포구,상암동,치료,NaN,0,특장차,뇌병
30639,30639,30641,2025-01-08 11:05:00.190,2025-01-08 13:10:00.000,NaN,NaN,NaN,2025-01-08 11:05:44.000,노원구,하계1동,도봉구,쌍문제4동,재활,NaN,0,특장차,뇌병
63617,63617,63619,2025-01-15 13:15:00.647,2025-01-15 15:20:00.000,NaN,NaN,NaN,2025-01-15 15:20:50.000,광진구,자양제1동,광진구,자양제4동,귀가,NaN,0,특장차,신장
63624,63624,63626,2025-01-15 13:15:00.647,2025-01-15 15:20:00.000,NaN,NaN,NaN,2025-01-15 15:20:50.000,중구,소공동,성북구,길음제2동,귀가,NaN,0,특장차,지체
86078,86078,86080,2025-01-21 04:55:00.167,2025-01-21 07:00:00.000,NaN,NaN,NaN,2025-01-21 04:55:44.000,금천구,시흥제5동,영등포구,영등포동,치료,NaN,0,특장차,지체
86081,86081,86083,2025-01-21 04:55:00.167,2025-01-21 07:00:00.000,NaN,NaN,NaN,2025-01-21 04:55:44.000,영등포구,신길제6동,용산구,이태원제1동,통학/출근,NaN,0,특장차,지체
155445,155445,155447,2025-02-07 10:45:01.003,2025-02-07 12:50:00.000,NaN,NaN,NaN,2025-02-07 10:46:34.000,구로구,구로제1동,동작구,신대방제2동,재활,NaN,0,특장차,뇌병
155446,155446,155448,2025-02-07 10:45:01.003,2025-02-07 12:50:00.000,NaN,NaN,NaN,2025-02-07 10:46:34.000,양천구,신정3동,안양시동안구,비산1동,귀가,NaN,0,특장차,뇌병


### 주요 식별 컬럼 기준 중복 데이터 확인

시간 정보가 같더라도 출발지, 목적지, 이용목적, 차량구분, 장애유형이 다르면 서로 다른 이용 기록일 수 있다. 따라서 접수·예정·배차·승차·하차·취소 시각과 출발/목적 지역, 이용목적, 차량구분, 장애유형, 요금, 승차거리까지 모두 같은 행이 중복 입력되었는지 확인한다.

In [50]:
# CHECK_DUPLICATED_IDENTIFYING_COLUMNS

# 주요 식별 컬럼 기준 중복 데이터 확인

identifying_columns = [
    '접수일시',
    '예정일시',
    '배차일시',
    '승차일시',
    '하차일시',
    '취소일시',
    '출발구',
    '출발동',
    '목적구',
    '목적동',
    '이용목적',
    '차량구분',
    '장애유형',
    '요금',
    '승차거리'
]

duplicated_identifying_columns = df_raw[
    df_raw.duplicated(subset=identifying_columns, keep=False)
].copy()

duplicated_identifying_columns.insert(0, '원본인덱스', duplicated_identifying_columns.index)
duplicated_identifying_columns.insert(1, 'CSV행번호', duplicated_identifying_columns.index + 2)

print(f'주요 식별 컬럼 기준 중복 행 수: {len(duplicated_identifying_columns):,}')

duplicated_identifying_columns.sort_values(identifying_columns).head(100)

주요 식별 컬럼 기준 중복 행 수: 0


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형


### 출발지 또는 목적지 정보가 없는 데이터 확인

`출발구`, `출발동`, `목적구`, `목적동` 중 하나라도 비어 있는 행은 지역 기반 수요 분석에서 위치 정보가 누락된 경우이므로, 제외 또는 별도 처리 필요성을 확인한다.

In [51]:
# CHECK_MISSING_LOCATION_COLUMNS

# 출발구/출발동/목적구/목적동 중 하나라도 없는 데이터 확인

location_columns = [
    '출발구',
    '출발동',
    '목적구',
    '목적동'
]

missing_location_columns = df_raw[
    df_raw[location_columns].isna().any(axis=1)
].copy()

missing_location_columns.insert(0, '원본인덱스', missing_location_columns.index)
missing_location_columns.insert(1, 'CSV행번호', missing_location_columns.index + 2)

print(f'출발지 또는 목적지 정보가 없는 행 수: {len(missing_location_columns):,}')

missing_location_columns[
    [
        '원본인덱스',
        'CSV행번호',
        '접수일시',
        '예정일시',
        '배차일시',
        '승차일시',
        '하차일시',
        '취소일시',
        '출발구',
        '출발동',
        '목적구',
        '목적동',
        '이용목적',
        '요금',
        '승차거리',
        '차량구분',
        '장애유형'
    ]
]

출발지 또는 목적지 정보가 없는 행 수: 1


,원본인덱스,CSV행번호,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
234966,234966,234968,2025-02-25 09:40:31.000,2025-02-25 09:41:00.000,NaN,NaN,NaN,2025-02-25 09:40:34.000,NaN,NaN,NaN,NaN,기타,NaN,0,특장차,뇌병


### 이용목적, 차량구분, 장애유형 값 분포 확인

`이용목적`, `차량구분`, `장애유형`은 주요 범주형 분석 컬럼이므로, 고유값과 빈도를 확인해 오타, 공백, 미상 값, 과도하게 희소한 값이 있는지 점검한다.

In [52]:
# CHECK_CATEGORY_VALUE_COUNTS

# 이용목적, 차량구분, 장애유형의 unique 값과 빈도 확인

category_columns = [
    '이용목적',
    '차량구분',
    '장애유형'
]

for col in category_columns:
    value_counts = (
        df_raw[col]
        .fillna('NaN')
        .value_counts(dropna=False)
        .reset_index()
    )
    value_counts.columns = [col, '행 수']

    print(f'[{col}] unique 값 수: {df_raw[col].nunique(dropna=False):,}')
    display(value_counts)

[이용목적] unique 값 수: 17


,이용목적,행 수
0,기타,1338521
1,귀가,132585
2,치료,101923
3,예약기타,52828
4,재활,38444
5,통학/출근,23450
6,예약치료,11746
7,예약광역,7349
8,종교,6666
9,예약재활,4647


[차량구분] unique 값 수: 2


,차량구분,행 수
0,특장차,1393609
1,임차택시,328881


[장애유형] unique 값 수: 18


,장애유형,행 수
0,뇌병,844028
1,지체,575954
2,지적,103840
3,신장,91603
4,자폐,79014
5,시각,7526
6,일시,6571
7,호흡,3904
8,정신,2485
9,국가,2442


In [53]:
df_raw.shape

(1722490, 15)

## 정제된 탑승내역 CSV 저장

전처리 과정에서 이상 행을 제외한 `df_raw`를 분석용 정제 데이터로 저장한다. 원본 CSV는 수정하지 않고, 정제 결과는 `data/processed` 폴더에 별도 CSV 파일로 저장한다.

In [54]:
# SAVE_CLEANED_RIDE_HISTORY_CSV

# 정제된 df_raw를 data/processed에 CSV로 저장

processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / '서울시설공단_장애인콜택시 탑승내역_정제_20251231.csv'

df_raw.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
print(f'저장 행 수: {len(df_raw):,}')
print(f'저장 컬럼 수: {df_raw.shape[1]:,}')

df_raw.head()

저장 완료: /Users/blaumonde/calltaxi-DA/data/processed/서울시설공단_장애인콜택시 탑승내역_정제_20251231.csv
저장 행 수: 1,722,490
저장 컬럼 수: 15


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,이용목적,요금,승차거리,차량구분,장애유형
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,기타,3400.0,17667,특장차,지체
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,기타,NaN,0,특장차,뇌병
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,기타,1500.0,2910,특장차,뇌병
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,기타,2900.0,10421,특장차,지체
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,기타,3000.0,11943,특장차,지체


## 정제 후 접수일시 파생 컬럼 CSV 생성

정제된 `df_raw`를 기준으로 접수일시 파생 컬럼을 생성하고 CSV로 저장한다.

In [55]:
# CREATE_REQUEST_DERIVED_CSV_AFTER_CLEANING
output_path = PROJECT_ROOT / 'data' / 'processed' / '서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv'

weekday_names = {
    0: '월요일',
    1: '화요일',
    2: '수요일',
    3: '목요일',
    4: '금요일',
    5: '토요일',
    6: '일요일',
}

request_dt = pd.to_datetime(df_raw['접수일시'], errors='coerce')
iso_calendar = request_dt.dt.isocalendar()

df_request_preprocessed = df_raw.copy()
df_request_preprocessed['접수일자'] = request_dt.dt.strftime('%Y-%m-%d')
df_request_preprocessed['접수시간'] = request_dt.dt.strftime('%H:%M:%S')
df_request_preprocessed['접수시간대'] = request_dt.dt.hour.astype('Int64')
df_request_preprocessed['접수요일'] = request_dt.dt.dayofweek.map(weekday_names)
df_request_preprocessed['접수월'] = request_dt.dt.to_period('M').astype(str)
df_request_preprocessed['접수ISO주차'] = iso_calendar.week.astype('Int64')

month_start = request_dt.dt.to_period('M').dt.start_time
month_week = ((request_dt.dt.day + month_start.dt.dayofweek - 1) // 7 + 1).astype('Int64')
df_request_preprocessed['접수월주차'] = request_dt.dt.strftime('%Y-%m') + '-' + month_week.astype(str) + '주차'

df_request_preprocessed.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
print(f'저장 행 수: {len(df_request_preprocessed):,}')
df_request_preprocessed.head()


저장 완료: /Users/blaumonde/calltaxi-DA/data/processed/서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv
저장 행 수: 1,722,490


,접수일시,예정일시,배차일시,승차일시,하차일시,취소일시,출발구,출발동,목적구,목적동,...,승차거리,차량구분,장애유형,접수일자,접수시간,접수시간대,접수요일,접수월,접수ISO주차,접수월주차
0,2025-01-01 00:01:26.127,2025-01-01 00:01:26.127,2025-01-01 00:27:31.307,2025-01-01 00:49:52.967,2025-01-01 01:35:06.747,NaN,용산구,남영동,강북구,수유제2동,...,17667,특장차,지체,2025-01-01,00:01:26,0,수요일,2025-01,1,2025-01-1주차
1,2025-01-01 00:03:10.557,2025-01-01 00:03:10.557,2025-01-01 00:14:39.800,NaN,NaN,2025-01-01 00:21:15.000,영등포구,여의동,강서구,화곡제6동,...,0,특장차,뇌병,2025-01-01,00:03:10,0,수요일,2025-01,1,2025-01-1주차
2,2025-01-01 00:03:31.230,2025-01-01 00:04:00.000,2025-01-01 00:19:41.900,2025-01-01 00:36:18.433,2025-01-01 00:49:45.857,NaN,노원구,상계5동,노원구,하계1동,...,2910,특장차,뇌병,2025-01-01,00:03:31,0,수요일,2025-01,1,2025-01-1주차
3,2025-01-01 00:03:40.573,2025-01-01 00:04:00.000,2025-01-01 00:33:41.650,2025-01-01 00:53:48.440,2025-01-01 01:33:38.120,NaN,영등포구,당산제1동,중구,명동,...,10421,특장차,지체,2025-01-01,00:03:40,0,수요일,2025-01,1,2025-01-1주차
4,2025-01-01 00:04:15.077,2025-01-01 00:04:15.077,2025-01-01 00:06:33.790,2025-01-01 00:40:07.840,2025-01-01 01:10:15.110,NaN,송파구,오금동,서초구,내곡동,...,11943,특장차,지체,2025-01-01,00:04:15,0,수요일,2025-01,1,2025-01-1주차


## 일별 이용현황 CSV 정리

일별 이용현황 원본 파일을 분석에 쓰기 쉬운 컬럼명과 날짜 순서로 정리해 CSV로 저장한다. 이 작업은 탑승내역 정제와 별개의 보조 데이터 정리 작업이다.


In [ ]:
import pandas as pd
from pathlib import Path

# 현재 노트북 위치가 notebooks/ 안이면 프로젝트 루트는 부모 폴더
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

matches = list(raw_dir.glob("*일별현황*.xls"))

if not matches:
    print("raw_dir:", raw_dir)
    print("raw 파일 목록:")
    for path in raw_dir.glob("*"):
        print(path.name)
    raise FileNotFoundError("일별현황 xls 파일을 찾지 못했습니다.")

xls_path = matches[0]
output_path = processed_dir / "서울시설공단_장애인콜택시 일별이용현황_20251231.csv"

processed_dir.mkdir(parents=True, exist_ok=True)

for enc in ["cp949", "euc-kr", "utf-8"]:
    try:
        df_taxi_daily = pd.read_html(str(xls_path), encoding=enc)[0]
        print(f"성공: {enc}")
        break
    except Exception as e:
        print(f"{enc} 실패: {e}")
else:
    raise RuntimeError("일별현황 xls 파일을 읽지 못했습니다.")

df_taxi_daily.columns = [
    "기준일",
    "차량운행",
    "접수건",
    "탑승건",
    "평균대기시간",
    "평균요금",
    "평균승차거리",
]

df_taxi_daily = df_taxi_daily.drop(index=0).reset_index(drop=True)
df_taxi_daily["기준일"] = pd.to_datetime(df_taxi_daily["기준일"], errors="coerce")
df_taxi_daily = df_taxi_daily.sort_values("기준일").reset_index(drop=True)

df_taxi_daily.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {output_path}")
df_taxi_daily.head()

성공: cp949
저장 완료: /Users/blaumonde/calltaxi-DA/data/processed/서울시설공단_장애인콜택시 일별이용현황_20251231.csv


,기준일,차량운행,접수건,탑승건,평균대기시간,평균요금,평균승차거리
0,2025-01-01,257,1437,1177,28,2252,8962
1,2025-01-02,665,5240,4525,24.8,2258,8119
2,2025-01-03,655,5445,4706,27.7,2231,8062
3,2025-01-04,407,2213,1910,21,2385,9616
4,2025-01-05,242,1844,1557,31.3,2359,9535
